# Workflow Test — Full Dealership Flow

This notebook demonstrates a complete `prompt-ml` workflow for an inbound auto-dealership phone call.

## What it shows

| Primitive | Used for |
|---|---|
| `Sequence` | Top-level call flow: Greeting → Branch → done |
| `Branch` | Route caller to Service, Sales, or Other based on intent |
| `Sequence` (nested) | Service sub-flow: Collect Address → Book → Confirm |
| `DataCollectionInstruction` | Collect caller's address with auto-confirm |
| `ClassifyInstruction` | Detect caller intent (rules + LLM) |
| `ActionInstruction` | Mock booking API call |
| `OutputInstruction` | Greeting, farewell, confirmation (static + template) |
| `Context` | Shared data — intent, address, booking — written and read across all steps |

## Flow structure

```
Sequence [
  Greeting                           (OutputInstruction)
  Branch [
    classifier: ClassifyIntent        (ClassifyInstruction)
    "service"  → Sequence [
                   CollectAddress    (DataCollectionInstruction)
                   BookService       (ActionInstruction)
                   ServiceConfirmation (OutputInstruction)
                 ]
    "sales"    → SalesFarewell       (OutputInstruction)
    "other"    → GenericFarewell     (OutputInstruction)
  ]
]
```

---

**How to use:**
1. Run **Setup** and **Define** cells once.
2. Run **Start** to get the opening message.
3. Edit `user_input` in the **Chat Turn** cell and re-run it for each message.
4. Run **Inspect** cells any time to see collected data and flow state.
5. Jump to **§ MockBackend Demo** at the bottom to see the full scenario replayed without an API key.

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
%pip install -q openai

In [1]:
# ── Cell 2: Imports & backend ─────────────────────────────────────────────────
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from prompt_ml import (
    
    Context, # chat context
    Field, # data collection

    ActionInstruction, 
    ActionResult,
    
    OutputInstruction, 
    DataCollectionInstruction, 
    ClassifyInstruction, 

    # flow library
    Branch, 
    Flow,
    Sequence,
)
from prompt_ml.backend.mock import MockBackend
from prompt_ml.backend.openai_backend import OpenAIBackend

API_KEY = "sk-proj-fw6fRj9ITkOgfYwn8O6RT3BlbkFJKZMBX9zNDq2nMmSd82Lp"

backend = OpenAIBackend(api_key=API_KEY, model="gpt-4o-mini", temperature=0.2)
print(f"Backend ready → {backend.model}")

Backend ready → gpt-4o-mini


---
## § Define Instructions
Each instruction is a plain Python class. No wiring needed — the workflow layer handles sequencing.

In [43]:
# ── Cell 3: Instruction definitions ──────────────────────────────────────────

# ── OutputInstruction: static greeting ───────────────────────────────────────
class Greeting(OutputInstruction):
    text = "Thank you for calling Sharma Dealership. How can I help you today?"


# ── ClassifyInstruction: intent detection ─────────────────────────────────────
# Rule-based first (zero LLM cost), falls back to LLM for ambiguous inputs.
class ClassifyIntent(ClassifyInstruction):
    labels = {
        "service": "If the user wants a service appointment request, or has a query or request that should be handled by the service team.",
        "sales":   "Buy a vehicle or get a price quote",
        "other":   "Anything else",
    }
    rules = {
        "service": ["oil change", "service", "repair", "brake", "tyre", "maintenance", "check", "alter"],
        "sales":   ["buy", "purchase", "new car", "price", "quote", "model"]
    }
    default_label = "other"
    output_key    = "intent"   # written to context as ctx["intent"]

    backend = OpenAIBackend(api_key=API_KEY)


# ── DataCollectionInstruction: address collection with auto-confirm ───────────
class CollectAddress(DataCollectionInstruction):
    context_namespace = "address"   # fields written as ctx["address.city"], etc.
    auto_confirm      = True         # after all fields: ask caller to confirm
    system_context    = (
        "You are a friendly phone assistant for Sharma Dealership. "
        "Keep responses concise — the caller is on the phone."
    )
    
    line1:    str = Field(required=True,  description="Street address line 1")
    line2:    str = Field(required=False, description="Apartment or suite (optional)")
    city:     str = Field(required=True,  description="City name")
    state:    str = Field(required=True,  description="State (two-letter abbreviation)")
    zip_code: str = Field(required=True,  description="Five-digit ZIP code")

    def on_complete(self) -> str:
        parts = [self.line1]
        if self.line2:
            parts.append(self.line2)
        parts += [self.city, f"{self.state} {self.zip_code}"]
        return f"Got it — {', '.join(parts)}."


# ── ActionInstruction: mock booking API call ──────────────────────────────────
class BookService(ActionInstruction):
    context_namespace = "booking"   # result written as ctx["booking.booking_id"], etc.

    def run(self) -> ActionResult:
        city = self.context.get("address.city", "your area")
        return ActionResult.ok(
            data={"booking_id": "SVC-001", "location": city},
            message=f"Your service appointment is confirmed in {city}.",
        )


# ── OutputInstruction: template — pulls from context set by BookService ───────
class ServiceConfirmation(OutputInstruction):
    template = (
        "All set! Booking reference: {booking.booking_id}. "
        "We look forward to seeing you at our {booking.location} location. Goodbye!"
    )


# ── OutputInstruction: branch farewell messages ───────────────────────────────
# class SalesFarewell(OutputInstruction):
#     text = "Connecting you to our sales team now. Please hold!"


class SalesFarewell(OutputInstruction):
    system_prompt = (
        "The user request has to redirected to the sales team. "
        "Generate a warm and conversational message to the user to redirect them to the sales team. The message should be under 10 words."
    )
    backend = OpenAIBackend(api_key=API_KEY)


class GenericFarewell(OutputInstruction):
    text = "Thanks for calling Acme Auto. Have a wonderful day!"


print("All instructions defined.")

All instructions defined.


---
## § Define the Flow
`Flow.build()` wires the instructions together. Each instruction that needs LLM access or shared data receives `context` and `backend` here.

In [38]:
# ── Cell 4: Flow definition ───────────────────────────────────────────────────

class DealershipFlow(Flow):
    """
    Full inbound call handler.

    Sequence
      ├─ Greeting                   ← fires immediately (auto_advance)
      └─ Branch (ClassifyIntent)
           ├─ "service" → Sequence
           │      ├─ CollectAddress
           │      ├─ BookService    ← fires automatically after address (auto_advance)
           │      └─ ServiceConfirmation  ← fires automatically (auto_advance)
           ├─ "sales"  → SalesFarewell
           └─ "other"  → GenericFarewell
    """

    def build(self, context, backend):
        service_sequence = Sequence([
            CollectAddress(context=context, backend=backend),
            BookService(context=context),
            ServiceConfirmation(context=context),
        ])

        return Sequence([
            Greeting(),
            Branch(
                classifier=ClassifyIntent(context=context, backend=backend),
                routes={
                    "service": service_sequence,
                    "sales":   SalesFarewell(),
                    "other":   GenericFarewell(),
                },
                default="other",
            ),
        ])


print("DealershipFlow defined.")

DealershipFlow defined.


---
## § Run the Flow
Run **Start** once, then use the **Chat Turn** cell for each caller message.

In [50]:


flow = DealershipFlow(backend=backend)

opening = flow.start()
print(f"Agent: {opening}")

Agent: Thank you for calling Sharma Dealership. How can I help you today?


In [51]:

user_input = "I want to get my doors removed" 
# user_input = "221b baker street London KS 56010" 
# user_input = "yes" 

print(f"Caller: {user_input}")
reply = flow.execute(user_input)
print(f"Agent:  {reply}")

if flow.is_complete():
    print("\n[Flow complete]")

Caller: I want to get my doors removed
Agent:  Thanks for calling Acme Auto. Have a wonderful day!

[Flow complete]


---
## § Inspect

In [41]:
# ── Cell 7: Context snapshot ──────────────────────────────────────────────────
# Every instruction writes its data here. Watch it grow across turns.

snap = flow.context.snapshot()
if snap:
    print("Context:")
    for k, v in sorted(snap.items()):
        print(f"  {k}: {v!r}")
else:
    print("Context is empty (no data collected yet).")

Context:
  intent: 'other'


In [ ]:
# ── Cell 8: Flow state ────────────────────────────────────────────────────────

print(f"Flow complete:  {flow.is_complete()}")
print(f"Turns so far:   {len(flow.history)}")
print(f"Flow repr:      {flow!r}")

In [ ]:
# ── Cell 9: Conversation history ──────────────────────────────────────────────

print("=" * 50)
for i, turn in enumerate(flow.history):
    if turn["user"]:
        print(f"[{i}] Caller: {turn['user']}")
    if turn["agent"]:
        print(f"[{i}] Agent:  {turn['agent']}")
    print()

In [ ]:
# ── Cell 10: Reset (start a new call) ────────────────────────────────────────

flow.reset()
print("Flow reset. Context:", flow.context.snapshot())
print("History:", flow.history)

---
## § Why the live conversation didn't finish the appointment flow

When this notebook was run with a real OpenAI backend, something unexpected happened at turn 7:

```
[7] Caller: 12345
[7] Agent:  Perfect! I have all the information now. How can I assist you further with your oil change?
```

**What should have happened:**
After "12345" filled the last required field, `CollectAddress` (with `auto_confirm = True`) entered its **confirming phase** and called `_generate_confirm_prompt()`. That function made a free-form LLM call, which returned *"I have all the information now — how can I assist?"*. That's a helpful-sounding message, but it never asked *"Is that correct? Yes or no?"* — so the caller had no idea they needed to confirm.

**What did happen:**
Every message from turn 8 onward ("I need an appointment", "today", "12:30"…) hit `_handle_confirmation()` instead of advancing the flow. Each was classified as `"correction"` rather than `"confirmed"`, so `CollectAddress._phase` never reached `COMPLETE`. `is_complete()` returned `False`, the `Sequence` never advanced to `BookService` or `ServiceConfirmation`, and the LLM just improvised free-form responses.

**The fix (now applied):**
`_generate_confirm_prompt()` uses a **deterministic template** by default:
```
Just to confirm — here's what I have:
  line1: 742 Evergreen Terrace
  city: Springfield
  ...
Is that correct? Please say yes or no.
```
This always contains an unambiguous yes/no question, so `_classify_confirmation()` can reliably detect the caller's intent. Subclasses can opt back into LLM phrasing by setting `confirm_prompt_template = ""`.

---
## § MockBackend Demo

No API key needed. Runs a complete scripted conversation end-to-end in one shot to show the full flow structure.

Two scenarios are replayed:
1. **Service** — caller books a service appointment (uses every node in the flow)
2. **Sales** — caller is transferred to the sales team (Branch routes differently)

In [ ]:
# ── Cell 11: MockBackend — Service scenario ───────────────────────────────────
#
# ClassifyIntent uses RULES for "oil change" → no LLM call consumed.
# The confirm prompt is now a deterministic template — no LLM call needed there.
# Remaining scripted responses go to CollectAddress in order:
#   turn 1 extract, turn 1 ask_for, turn 2 extract, turn 3 classify-confirmation

service_mock = MockBackend(responses=[
    # Turn 1 — CollectAddress: extract from "I need an oil change" → nothing
    "{}",
    # Turn 1 — CollectAddress: ask for street address
    "Sure! Could I get your street address?",
    # Turn 2 — CollectAddress: extract all fields from address message
    '{"line1": "742 Evergreen Terrace", "city": "Springfield", "state": "IL", "zip_code": "62701"}',
    # Turn 2 — confirm prompt is now static (no LLM call here)
    # Turn 3 — CollectAddress: classify user's reply as confirmed
    "confirmed",
])

service_flow = DealershipFlow(backend=service_mock)

script = [
    "",                                               # start() — fires Greeting
    "I need an oil change",                           # → classified as service, asks for address
    "742 Evergreen Terrace, Springfield IL 62701",   # → all fields extracted, confirm prompt
    "Yes, that's correct",                            # → confirmed → book → farewell
]

print("═" * 60)
print("  SCENARIO: Service Appointment")
print("═" * 60)

for user_says in script:
    reply = service_flow.execute(user_says)
    if user_says:
        print(f"\nCaller: {user_says}")
    if reply:
        print(f"Agent:  {reply}")

print(f"\n[Complete: {service_flow.is_complete()}]")
print("\nFinal context:")
for k, v in sorted(service_flow.context.snapshot().items()):
    print(f"  {k}: {v!r}")

In [ ]:
# ── Cell 12: MockBackend — Sales scenario ─────────────────────────────────────
#
# "buy a car" matches the "sales" rule → no LLM call at all.
# Branch routes to SalesFarewell (auto_advance OutputInstruction) immediately.

sales_mock = MockBackend()  # no scripted responses needed — all rule-based

sales_flow = DealershipFlow(backend=sales_mock)

print("═" * 60)
print("  SCENARIO: Sales Enquiry")
print("═" * 60)

opening = sales_flow.start()
print(f"\nAgent:  {opening}")

user_says = "I want to buy a new car"
reply = sales_flow.execute(user_says)
print(f"Caller: {user_says}")
print(f"Agent:  {reply}")

print(f"\n[Complete: {sales_flow.is_complete()}]")
print(f"Intent classified as: {sales_flow.context.get('intent')!r}")
print(f"LLM calls made: {sales_mock.call_count}  ← zero, rules handled everything")

In [ ]:
# ── Cell 13: Context nested view ─────────────────────────────────────────────
# Show the final context as a nested dict (dotted keys expanded)

import json
print(json.dumps(service_flow.context.as_nested(), indent=2))